# Phase 3 — QEvasion response-clarity evaluation (Google Colab)

This notebook replaces the synthetic Phase 3 benchmark with the real, human-annotated `QAEvasion.csv` dataset. Run it in **Google Colab** with a GPU and Internet access. This is zero-shot inference/evaluation; do not train on the gold labels here.

**Persistence:** the notebook mounts Google Drive and writes checkpoints, predictions, metrics, and a results ZIP to Drive. Start with `smoke`, then `pilot`, and use `paper` only after the run configuration and model revisions are frozen.

**Inputs:** the model sees only `question` and `interview_answer`. The gold `label`, `gpt3.5_summary`, and `gpt3.5_prediction` columns are excluded from prompts to avoid label leakage.


## 1. Configure the run

Change `PROFILE` to `"smoke"`, `"pilot"`, or `"paper"`. The paper profile evaluates all annotation rows in the provided 3,448-row CSV. It may exceed a single free Colab session; resumable checkpoints allow a restarted session to continue. For publication-quality runs, pin each model to a Hugging Face commit SHA in `MODEL_REVISIONS`.


In [ ]:
# User-editable settings
PROFILE = "smoke"  # smoke=27 annotation rows; pilot=180; paper=all 3,448
MODEL_IDS = [
    "Qwen/Qwen2.5-1.5B-Instruct",
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
]
MODEL_REVISIONS = {model_id: None for model_id in MODEL_IDS}  # pin commit SHAs for final runs
SEED = 42
MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 16
CHECKPOINT_EVERY = 50
USE_4BIT = True
RESUME = True

REPO_URL = "https://github.com/MichealSK/political-bias-lab.git"
REPO_BRANCH = "master"
REPO_DIR = "/content/political-bias-lab"
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/political-bias-lab"


## 2. Mount Google Drive, prepare the repository, and install dependencies

The notebook clones the repository into the temporary Colab runtime and uses `requirements-colab.txt`. Outputs are written to Drive so they survive runtime resets. If you have already cloned the repository during this session, the notebook pulls the latest `master` commit before running.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
repo_dir = Path(REPO_DIR)
if not (repo_dir / ".git").exists():
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)])
else:
    subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", REPO_BRANCH])

os.chdir(repo_dir)
if not (repo_dir / "src" / "phase3_evasion.py").exists():
    raise FileNotFoundError(
        "Phase 3 files are not present in the checked-out repository. Merge the Phase 3 Colab patch, "
        "then restart this notebook so it clones the updated master branch."
    )
DATA_PATH = repo_dir / "data" / "QAEvasion.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}; make sure data/QAEvasion.csv is committed to the repository.")

OUTPUT_DIR = Path(DRIVE_PROJECT_DIR) / "results" / "phase3_qaevasion"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_dir / "requirements-colab.txt")])
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
print("Repository:", repo_dir)
print("Dataset:", DATA_PATH)
print("Persistent results:", OUTPUT_DIR)


## 3. Validate the dataset before spending GPU time

This checks the schema, label inventory, number of annotation rows, unique question-answer inputs, interviews, and presidents.


In [ ]:
import pandas as pd
from src.phase3_evasion import load_qaevasion_csv, sample_annotations, unique_inference_items, LABELS

df = load_qaevasion_csv(DATA_PATH)
print("Annotation rows:", len(df))
print("Unique question-answer items:", df["item_id"].nunique())
print("Interviews:", df["url"].nunique())
print("Presidents:", sorted(df["president"].unique().tolist()))
print("Annotators:", sorted(df["annotator_id"].astype(str).unique().tolist()))
display(df["label"].value_counts().reindex(LABELS).rename("count").to_frame())
assert len(df) == 3448, "Unexpected dataset row count; inspect the input before continuing."
assert set(df["label"].unique()) == set(LABELS)


## 4. Confirm the Colab GPU is enabled

In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU) before running inference. Model inference is sequential to reduce VRAM use.


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Enable a GPU in Google Colab: Runtime → Change runtime type → GPU.")


## 5. Run zero-shot classification

Each model is loaded and unloaded sequentially. Checkpoints are stored under the persistent Drive results directory, so rerunning with `RESUME = True` reuses completed predictions. The smoke profile is deliberately small and includes all nine labels.


In [ ]:
from src.phase3_evasion import run_phase3_qaevasion
import gc, json

all_reports = {}
for model_id in MODEL_IDS:
    print(f"\n===== Phase 3 model: {model_id} | profile: {PROFILE} =====")
    report = run_phase3_qaevasion(
        input_path=DATA_PATH,
        output_dir=OUTPUT_DIR,
        model_id=model_id,
        revision=MODEL_REVISIONS.get(model_id),
        profile=PROFILE,
        seed=SEED,
        max_input_tokens=MAX_INPUT_TOKENS,
        max_new_tokens=MAX_NEW_TOKENS,
        checkpoint_every=CHECKPOINT_EVERY,
        use_4bit=USE_4BIT,
        resume=RESUME,
    )
    all_reports[model_id] = report
    print(json.dumps({
        "n_annotation_rows_sampled": report["n_annotation_rows_sampled"],
        "n_unique_items_sampled": report["n_unique_items_sampled"],
        "prediction_coverage": report["prediction_coverage"],
        "fine_grained_macro_f1": report["fine_grained_9_way"]["macro_f1"],
        "clarity_macro_f1": report["clarity_3_way"]["macro_f1"],
    }, indent=2))
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 6. Compare the models and save a summary to Drive


In [ ]:
summary_rows = []
for model_id, report in all_reports.items():
    summary_rows.append({
        "model_id": model_id,
        "profile": report["profile"],
        "annotation_rows": report["n_annotation_rows_sampled"],
        "unique_items": report["n_unique_items_sampled"],
        "coverage": report["prediction_coverage"],
        "9way_accuracy": report["fine_grained_9_way"]["accuracy"],
        "9way_macro_f1": report["fine_grained_9_way"]["macro_f1"],
        "9way_weighted_f1": report["fine_grained_9_way"]["weighted_f1"],
        "3way_accuracy": report["clarity_3_way"]["accuracy"],
        "3way_macro_f1": report["clarity_3_way"]["macro_f1"],
    })
summary = pd.DataFrame(summary_rows)
display(summary)
summary_path = OUTPUT_DIR / f"phase3_qaevasion_model_comparison_{PROFILE}.csv"
summary.to_csv(summary_path, index=False)
print("Saved summary:", summary_path)


## 7. Package results

The ZIP is saved next to the results folder in Google Drive. It includes predictions, annotation-level metrics, confusion matrices, and checkpoints. You can optionally download it from the Colab session.


In [ ]:
import shutil
archive = shutil.make_archive(
    str(OUTPUT_DIR.parent / f"phase3_qaevasion_results_{PROFILE}"),
    "zip",
    root_dir=OUTPUT_DIR,
)
print("Results archive:", archive)

DOWNLOAD_RESULTS_ZIP = False  # set True if you also want a browser download
if DOWNLOAD_RESULTS_ZIP:
    from google.colab import files
    files.download(archive)


## Method and interpretation notes

- Fine-grained evaluation uses the nine QEvasion labels. A secondary mapping reports three clarity categories: `1.1 Explicit` → Clear Reply; `1.2 Implicit`, `2.1 Dodging`, `2.2 Deflection`, `2.3 Partial/half-answer`, and `2.4 General` → Ambivalent; `2.6 Declining to answer`, `2.7 Claims ignorance`, and `2.8 Clarification` → Clear Non-Reply.
- Multiple annotations for the same question-answer item are retained for annotation-level scoring. Inference is performed once per unique item.
- Invalid/unparseable model outputs are not forced into a class; they reduce prediction coverage and are reported.
- Do not describe this 3,448-row file as the official held-out test split. Keep the dataset citation and license terms in `PHASE3_QAEVASION.md` with any public release.
